<a href="https://colab.research.google.com/github/ranjiparker/vst-animated/blob/main/captions_key_part1_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Extracting Keywords with TF-IDF and Python’s Scikit-Learn

In [ ]:
import pandas as pd

# read json into a dataframe
df_idf=pd.read_json("/content/captions_abstract_v002_val2015.json",lines=True)

# print schema
print("Schema:\n\n",df_idf.dtypes)
print("Number of questions,columns=",df_idf.shape)

Schema:

 image_id     int64
id           int64
caption     object
dtype: object
Number of questions,columns= (50000, 3)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import re
def pre_process(text):
    
    # lowercase
    text=text.lower()
    
    #remove tags
    text=re.sub("","",text)
    
    # remove special characters and digits
    text=re.sub("(\\d|\\W)+"," ",text)
    
    return text

df_idf['text'] = df_idf['caption']
df_idf['text'] = df_idf['text'].apply(lambda x:pre_process(x))

#show the second 'text' just for fun
df_idf['text'][2]

'a boy sitting on a stool in his living room calls his dog over to him '

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import re

def get_stop_words(stop_file_path):
    """load stop words """
    
    with open(stop_file_path, 'r', encoding="utf-8") as f:
        stopwords = f.readlines()
        stop_set = set(m.strip() for m in stopwords)
        return frozenset(stop_set)

#load a set of stop words
stopwords=get_stop_words("/content/NLTK's list of english stopwords")

#get the text column 
docs=df_idf['text'].tolist()

#create a vocabulary of words, 
#ignore words that appear in 85% of documents, 
#eliminate stop words
cv=CountVectorizer(max_df=0.85,stop_words=stopwords)
word_count_vector=cv.fit_transform(docs)


In [ ]:
cv=CountVectorizer(max_df=0.85,stop_words=stopwords,max_features=10000)
word_count_vector=cv.fit_transform(docs)


In [ ]:
list(cv.vocabulary_.keys())[:20]

['boy',
 'small',
 'dog',
 'sitting',
 'front',
 'fireplace',
 'sits',
 'plays',
 'stool',
 'living',
 'room',
 'calls',
 'little',
 'chair',
 'fire',
 'side',
 'runs',
 'toward',
 'smiling',
 'two']

In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer

tfidf_transformer=TfidfTransformer(smooth_idf=True,use_idf=True)
tfidf_transformer.fit(word_count_vector)

TfidfTransformer()

In [ ]:
# read test docs into a dataframe and concatenate caption
df_test=pd.read_json("/captions_abstract_v002_val2015.json",lines=True)
df_test['text'] =  df_test['caption']
df_test['text'] =df_test['text'].apply(lambda x:pre_process(x))

# get test docs into a list
docs_test=df_test['text'].tolist()

In [ ]:
l=[]
for z in range(100): 
  # you only need to do this once, this is a mapping of index to 
  feature_names=cv.get_feature_names()

  # get the document that we want to extract keywords from
  doc=docs_test[z]

  #generate tf-idf for the given document
  tf_idf_vector=tfidf_transformer.transform(cv.transform([doc]))
  def sort_coo(coo_matrix):
      tuples = zip(coo_matrix.col, coo_matrix.data)
      return sorted(tuples, key=lambda x: (x[1], x[0]), reverse=True)
  #sort the tf-idf vectors by descending order of scores
  sorted_items=sort_coo(tf_idf_vector.tocoo())
  def extract_topn_from_vector(feature_names, sorted_items, topn=10):
      """get the feature names and tf-idf score of top n items"""
      
      #use only top n items from vector
      sorted_items = sorted_items[:topn]

      score_vals = []
      feature_vals = []
      
      # word index and corresponding tf-idf score
      for idx, score in sorted_items:
          
          #keep track of feature name and its corresponding score
          score_vals.append(round(score, 3))
          feature_vals.append(feature_names[idx])

      #create a tuples of feature,score
      #results = zip(feature_vals,score_vals)
      results= {}
      for idx in range(len(feature_vals)):
          results[feature_vals[idx]]=score_vals[idx]
      
      return results
  #extract only the top n; n here is 10
  keywords=extract_topn_from_vector(feature_names,sorted_items,10)


  for k in keywords:
      (k,keywords[k])
      l.append(k)
print(l)



/usr/local/lib/python3.7/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is deprecated in 1.0 and will be removed in 1.2. Please use get_feature_names_out instead.
  warnings.warn(msg, category=FutureWarning)
/usr/local/lib/python3.7/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is deprecated in 1.0 and will be removed in 1.2. Please use get_feature_names_out instead.
  warnings.warn(msg, category=FutureWarning)
/usr/local/lib/python3.7/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is deprecated in 1.0 and will be removed in 1.2. Please use get_feature_names_out instead.
  warnings.warn(msg, category=FutureWarning)
/usr/local/lib/python3.7/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is d

['fireplace', 'small', 'front', 'dog', 'boy', 'sitting', 'plays', 'fireplace', 'front', 'sits', 'dog', 'boy', 'calls', 'stool', 'living', 'room', 'dog', 'boy', 'sitting', 'side', 'fire', 'front', 'chair', 'little', 'dog', 'boy', 'sitting', 'runs', 'smiling', 'toward', 'fireplace', 'front', 'dog', 'boy', 'ladies', 'toys', 'play', 'baby', 'watching', 'two', 'buy', 'sit', 'floor', 'young', 'couch', 'playing', 'girl', 'woman', 'dollhouse', 'toddler', 'television', 'watches', 'couple', 'plays', 'family', 'brown', 'tv', 'watching', 'couch', 'sitting', 'watched', 'played', 'tv', 'floor', 'boy', 'sunny', 'lady', 'day', 'old', 'little', 'park', 'girl', 'younger', 'walk', 'slide', 'older', 'pond', 'near', 'girl', 'woman', 'grandmother', 'walking', 'young', 'park', 'girl', 'grandaughter', 'taking', 'grandma', 'slide', 'pond', 'held', 'grandmother', 'hand', 'slide', 'girl', 'look', 'young', 'soccer', 'ball', 'plays', 'woman', 'older', 'man', 'talks', 'mother', 'soccer', 'child', 'plays', 'woman', 

/usr/local/lib/python3.7/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is deprecated in 1.0 and will be removed in 1.2. Please use get_feature_names_out instead.
  warnings.warn(msg, category=FutureWarning)
/usr/local/lib/python3.7/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is deprecated in 1.0 and will be removed in 1.2. Please use get_feature_names_out instead.
  warnings.warn(msg, category=FutureWarning)
/usr/local/lib/python3.7/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is deprecated in 1.0 and will be removed in 1.2. Please use get_feature_names_out instead.
  warnings.warn(msg, category=FutureWarning)
/usr/local/lib/python3.7/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is d

In [ ]:
import numpy
import sys
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM
from keras.utils import np_utils
from keras.callbacks import ModelCheckpoint

In [ ]:
def tokenize_words(input):
    # lowercase everything to standardize it
    input = input.lower()

    # instantiate the tokenizer
    tokenizer = RegexpTokenizer(r'\w+')
    tokens = tokenizer.tokenize(input)

    # if the created token isn't in the stop words, make it part of "filtered"
    filtered = filter(lambda token: token not in stopwords.words('english'), tokens)
    return " ".join(filtered)

In [ ]:
chars = sorted(list(set(l)))
char_to_num = dict((c, i) for i, c in enumerate(chars))

In [ ]:
input_len = len(l)
vocab_len = len(chars)
print ("Total number of characters:", input_len)
print ("Total vocab:", vocab_len)

Total number of characters: 621
Total vocab: 214


In [ ]:
seq_length = 100
x_data = []
y_data = []

In [ ]:
# loop through inputs, start at the beginning and go until we hit
# the final character we can create a sequence out of
for i in range(0, input_len - seq_length, 1):
    # Define input and output sequences
    # Input is the current character plus desired sequence length
    in_seq = l[i:i + seq_length]

    # Out sequence is the initial character plus total sequence length
    out_seq = l[i + seq_length]

    # We now convert list of characters to integers based on
    # previously and add the values to our lists
    x_data.append([char_to_num[char] for char in in_seq])
    y_data.append(char_to_num[out_seq])

In [ ]:
n_patterns = len(x_data)
print ("Total Patterns:", n_patterns)

Total Patterns: 521


In [ ]:
X = numpy.reshape(x_data, (n_patterns, seq_length, 1))
X = X/float(vocab_len)

In [ ]:
y = np_utils.to_categorical(y_data)

In [ ]:
model = Sequential()
model.add(LSTM(256, input_shape=(X.shape[1], X.shape[2]), return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(256, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(128))
model.add(Dropout(0.2))
model.add(Dense(y.shape[1], activation='softmax'))

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam')

In [ ]:
filepath = "model_weights_saved.hdf5"
checkpoint = ModelCheckpoint(filepath, monitor='loss', verbose=1, save_best_only=True, mode='min')
desired_callbacks = [checkpoint]

In [ ]:
model.fit(X, y, epochs=20, batch_size=256, callbacks=desired_callbacks)

Epoch 1/20
3/3 [==============================] - ETA: 0s - loss: 5.3554
Epoch 00001: loss improved from inf to 5.35540, saving model to model_weights_saved.hdf5
3/3 [==============================] - 15s 2s/step - loss: 5.3554
Epoch 2/20
3/3 [==============================] - ETA: 0s - loss: 5.2821
Epoch 00002: loss improved from 5.35540 to 5.28212, saving model to model_weights_saved.hdf5
3/3 [==============================] - 9s 2s/step - loss: 5.2821
Epoch 3/20
3/3 [==============================] - ETA: 0s - loss: 5.1154
Epoch 00003: loss improved from 5.28212 to 5.11539, saving model to model_weights_saved.hdf5
3/3 [==============================] - 9s 2s/step - loss: 5.1154
Epoch 4/20
3/3 [==============================] - ETA: 0s - loss: 5.0234
Epoch 00004: loss improved from 5.11539 to 5.02344, saving model to model_weights_saved.hdf5
3/3 [==============================] - 9s 2s/step - loss: 5.0234
Epoch 5/20
3/3 [==============================] - ETA: 0s - loss: 4.9384
Epoch 

In [ ]:
filename = "model_weights_saved.hdf5"
model.load_weights(filename)
model.compile(loss='categorical_crossentropy', optimizer='adam')

In [ ]:
num_to_char = dict((i, c) for i, c in enumerate(chars))

In [ ]:
start = numpy.random.randint(0, len(x_data) - 1)
pattern = x_data[start]
print("Random Seed:")
print("\"", ''.join([num_to_char[value]+" " for value in pattern]), "\"") 

Random Seed:
" man woman dinner together couple dog sitting snacks share begging eating couple table young dog dinner eating couple table sitting mushroom covered playing park man woman around outside playing man woman tossing baseball man woman growing wild mushrooms pick going field mushrooms field ground man woman guys camp ground fire next two sitting warming men beside fire two keeping warm campfire men grass two break taking baseball boys fire playing two burning boys fire two park jump rope grass holding sitting woman knees rope grass holding woman kneeling jump rope grass woman partly cloudy rope jumping outside day woman roping  "
